In [1]:
names = open('names.txt', 'r').read().splitlines()
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [2]:
vocab = sorted(list(set(''.join(names))))
vocab.append('<S>')
vocab.append('<PAD>')
token_to_index = {c : i for i, c in enumerate(vocab)}
index_to_token = {i : c for c, i in token_to_index.items()}

print(len(vocab))

28


In [3]:
vocab_size = len(vocab)
hidden_size = 128

### Building the datasets

In [4]:
import torch
import torch.nn.functional as F
g = torch.Generator().manual_seed(10000)

def dataset_builder(data):
    X, Y = [], []
    
    for item in data:
        tokens = ['<S>'] + list(item) + ['<S>']
        indices = [token_to_index[token] for token in tokens]
        X.append(indices[:-1])
        Y.append(indices[1:])
    
    return X, Y

import random
random.seed(123)
random.shuffle(names)

n1 = int(0.8 * len(names))
n2 = int(0.9 * len(names))
         
xtr, ytr = dataset_builder(names[:n1])
X_dev, Y_dev = dataset_builder(names[n1:n2])
X_test, Y_test = dataset_builder(names[n2:])

len(xtr), len(ytr)

(25626, 25626)

### Defining the RNN cell

In [5]:
import torch.nn as nn

class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wx = nn.Linear(input_size, hidden_size)
        self.Wh = nn.Linear(hidden_size, hidden_size)
        self.act = nn.Tanh()
        
    def forward(self, x, h):
        h = self.act(self.Wh(h) + self.Wx(x))
        return h

### Defining the model

In [6]:
class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_size)
        self.rnn = RNNCell(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size
        
    def forward(self, x, h=None):
        if h is None:
            h = self.init_hidden(x.shape[0])
        
        x = self.emb(x)
        
        batch_logits = []
        for i in range(x.shape[1]):
            h = self.rnn(x[:, i : i + 1], h)
            token_logits = self.fc(h)
            batch_logits.append(token_logits)
        
        batch_logits = torch.cat(batch_logits, dim=1)
        
        return batch_logits, h
           
    def init_hidden(self, batch_size):
        return torch.zeros((batch_size, 1, self.hidden_size))

### padding according to the max length in the batch

In [7]:
def pad(xbat, ybat):
    max_length = max(len(item) for item in xbat)
    # print(max_length)
    x_pad = []
    y_pad = []
    for itemx, itemy in zip(xbat, ybat):
        x_pad.append(itemx + [token_to_index['<PAD>']] * (max_length - len(itemx)))
        y_pad.append(itemy + [token_to_index['<PAD>']] * (max_length - len(itemy)))
        
    return torch.tensor(x_pad), torch.tensor(y_pad)

xp, yp = pad(xtr[:4], ytr[:4])

print(yp.view(-1))

for x in xp:
    print([index_to_token[i.item()] for i in x])

tensor([11, 20,  0, 13, 13, 26, 27, 27, 27, 27, 18,  7,  0,  8, 13, 26, 27, 27,
        27, 27, 17, 20, 15,  4, 17, 19, 26, 27, 27, 27, 12, 14, 10, 18,  7,  0,
         6, 13,  0, 26])
['<S>', 'l', 'u', 'a', 'n', 'n', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
['<S>', 's', 'h', 'a', 'i', 'n', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
['<S>', 'r', 'u', 'p', 'e', 'r', 't', '<PAD>', '<PAD>', '<PAD>']
['<S>', 'm', 'o', 'k', 's', 'h', 'a', 'g', 'n', 'a']


### Training the model

In [8]:
model = RNNLM(vocab_size, hidden_size)
logits, h = model(xp, h=None)

In [9]:
lr = 0.001
num_epochs = 10
batch_size = 32

# number of parameters
sum(p.nelement() for p in model.parameters())

# loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=token_to_index['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr)

# training loop
for epoch in range(num_epochs):
    # Reshuffle the data
    perm = torch.randperm(len(xtr))
    xtr = [xtr[i] for i in perm]
    ytr = [ytr[i] for i in perm]
    
    model.train()
    total_loss = 0
    
    for i in range(0, len(xtr), batch_size):
        # batching
        xbat = xtr[i : i + batch_size]
        ybat = ytr[i : i + batch_size]
        
        # padding
        xp, yp = pad(xbat, ybat)
        
        # forward pass
        logits, h = model(xp)
        logits = logits.view(-1, vocab_size)
        yp = yp.view(-1)
        loss = criterion(logits, yp)
        
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss
    
    avg_loss = total_loss / (len(xtr) // batch_size)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')
    
    # evaluate
    eval_loss = 0
    model.eval()
    
    with torch.no_grad():
        for i in range(0, len(X_dev), batch_size):
            # batching
            xbat = X_dev[i : i + batch_size]
            ybat = Y_dev[i : i + batch_size]
                    
            # padding
            xp, yp = pad(xbat, ybat)
                    
            # forward pass
            logits, h = model(xp)
            logits = logits.view(-1, vocab_size)
            yp = yp.view(-1)
            loss = criterion(logits, yp)

            eval_loss += loss.item()
            
    avg_eval_loss = eval_loss / (len(X_dev) // batch_size)
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_eval_loss:.4f}')
        

Epoch [1/10], Loss: 2.2435
Epoch [1/10], Validation Loss: 2.1812
Epoch [2/10], Loss: 2.1236
Epoch [2/10], Validation Loss: 2.1276
Epoch [3/10], Loss: 2.0830
Epoch [3/10], Validation Loss: 2.1033
Epoch [4/10], Loss: 2.0565
Epoch [4/10], Validation Loss: 2.0871
Epoch [5/10], Loss: 2.0368
Epoch [5/10], Validation Loss: 2.0827
Epoch [6/10], Loss: 2.0222
Epoch [6/10], Validation Loss: 2.0693
Epoch [7/10], Loss: 2.0095
Epoch [7/10], Validation Loss: 2.0689
Epoch [8/10], Loss: 1.9995
Epoch [8/10], Validation Loss: 2.0559
Epoch [9/10], Loss: 1.9911
Epoch [9/10], Validation Loss: 2.0563
Epoch [10/10], Loss: 1.9831
Epoch [10/10], Validation Loss: 2.0519


### Sampling from the model

In [10]:
def sample(model, context, max_length=100):
    model.eval()
    output = []
    with torch.no_grad():
        x = torch.tensor([[token_to_index['<S>']] + context])
        h = None
        for _ in range(max_length):
            logits, h = model(x, h)
            logits = logits[0, -1].softmax(dim=0)
            index = torch.multinomial(logits, 1)
            token = index_to_token[index.item()]
            
            if token == "<S>": break
            
            output.append(token)
            x = index.view(1, 1)
            
    return ''.join(output)


In [11]:
for i in range(10):
    print(sample(model, []))

maddier
eash
quittlei
lucke
sanvelly
witli
raisyn
yannah
issamsa
mariah


### Conditional sampling

In [12]:
prompt = 'k'
for i in range(10):
    out = sample(model, [token_to_index[tok] for tok in prompt])
    print(prompt + out)

kaliza
kimfrena
kyel
katin
kwylee
khasir
kingin
kaptin
knix
khalid


### Manually implementing the forward pass

In [55]:
# lookup table
C = torch.randn((vocab_size, hidden_size), generator=g)

# input -> hidden
Wx = torch.randn((hidden_size, hidden_size), generator=g) * 0.01

# previous hidden -> hidden
Wh = torch.randn((hidden_size, hidden_size), generator=g) * 0.01
b = torch.randn((hidden_size), generator=g) * 0.01

# hidden -> output
Wy = torch.randn((hidden_size, vocab_size), generator=g) *0.01
by = torch.zeros(vocab_size)

# initial hidden state
h0 = torch.ones(hidden_size) * 0.1
print(h0.shape)

print(f'{torch.tensor(xtr[:1]).shape = }')
h = torch.tanh(h0.matmul(Wh) + C[torch.tensor(xtr[:1])].matmul(Wx) + b)
print(h.shape)


logits = h @ Wy + by

counts = logits.exp()
probs = counts / counts.sum(dim=2, keepdim=True)
print(probs.shape)

print(probs[0, 0].sum())

loss = criterion(logits.view(-1, vocab_size), torch.tensor(ytr[:1]).view(-1))
loss

torch.Size([128])
torch.tensor(xtr[:1]).shape = torch.Size([1, 8])
torch.Size([1, 8, 128])
torch.Size([1, 8, 28])
tensor(1.0000)


tensor(3.3317)

### Fully PyTorchifying

In [91]:
x = torch.randn((32, 6, 64))
x.transpose(0, 1).shape

torch.Size([6, 32, 64])

In [ ]:
class RecurrentLM(nn.Module):
    def __init__(self, hidden_size, vocab_size, emb_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_size)
        self.rnn = nn.RNN(emb_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x):   
        x_emb = self.emb(x)
            
        output, hx = self.rnn(x_emb)
             
        logits = self.out(output)
        
        return logits, hx

In [121]:
model = RecurrentLM(hidden_size, vocab_size, hidden_size)
logits, h = model(xp)

In [122]:
lr = 0.001
num_epochs = 10
batch_size = 32

# number of parameters
sum(p.nelement() for p in model.parameters())

# loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=token_to_index['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr)

# training loop
for epoch in range(num_epochs):
    # Reshuffle the data
    perm = torch.randperm(len(xtr))
    xtr = [xtr[i] for i in perm]
    ytr = [ytr[i] for i in perm]
    
    model.train()
    total_loss = 0
    
    for i in range(0, len(xtr), batch_size):
        # batching
        xbat = xtr[i : i + batch_size]
        ybat = ytr[i : i + batch_size]
        
        # padding
        xp, yp = pad(xbat, ybat)
        
        # forward pass
        logits, h = model(xp)
        logits = logits.view(-1, vocab_size)
        yp = yp.view(-1)
        loss = criterion(logits, yp)
        
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss
    
    avg_loss = total_loss / (len(xtr) // batch_size)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')
    
    # evaluate
    eval_loss = 0
    model.eval()
    
    with torch.no_grad():
        for i in range(0, len(X_dev), batch_size):
            # batching
            xbat = X_dev[i : i + batch_size]
            ybat = Y_dev[i : i + batch_size]
                    
            # padding
            xp, yp = pad(xbat, ybat)
                    
            # forward pass
            logits, h = model(xp)
            logits = logits.view(-1, vocab_size)
            yp = yp.view(-1)
            loss = criterion(logits, yp)

            eval_loss += loss.item()
            
    avg_eval_loss = eval_loss / (len(X_dev) // batch_size)
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_eval_loss:.4f}')
        

Epoch [1/10], Loss: 2.2438
Epoch [1/10], Validation Loss: 2.1529
Epoch [2/10], Loss: 2.0935
Epoch [2/10], Validation Loss: 2.0968
Epoch [3/10], Loss: 2.0445
Epoch [3/10], Validation Loss: 2.0721
Epoch [4/10], Loss: 2.0124
Epoch [4/10], Validation Loss: 2.0478
Epoch [5/10], Loss: 1.9857
Epoch [5/10], Validation Loss: 2.0377
Epoch [6/10], Loss: 1.9637
Epoch [6/10], Validation Loss: 2.0282
Epoch [7/10], Loss: 1.9462
Epoch [7/10], Validation Loss: 2.0168
Epoch [8/10], Loss: 1.9300
Epoch [8/10], Validation Loss: 2.0103
Epoch [9/10], Loss: 1.9159
Epoch [9/10], Validation Loss: 2.0081
Epoch [10/10], Loss: 1.9037
Epoch [10/10], Validation Loss: 2.0019
